In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data import load_trips

trips = load_trips()
train = trips[trips["split"] == "train"]
valid = trips[trips["split"] == "valid"]

print(train.shape, valid.shape)
print(train["is_crawl"].mean(), valid["is_crawl"].mean())

(190878, 25) (190003, 25)
0.1970158949695617 0.19036015220812302


In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, average_precision_score

results = []


def evaluate(name, y_true, y_pred, y_score):
    pr_auc = average_precision_score(y_true, y_score)
    results.append({
        "model": name,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred),
        "pr_auc": pr_auc,
        "lift": pr_auc / y_true.mean(),
    })
    return pd.DataFrame(results)

In [3]:
y_valid = valid["is_crawl"]
zeros = np.zeros(len(valid), dtype=int)

evaluate("1. always no-crawl", y_valid, zeros, zeros)

,model,precision,recall,pr_auc,lift
0,1. always no-crawl,0.0,0.0,0.19036,1.0


In [4]:
pickup = train["tpep_pickup_datetime"]

by_hour = train.groupby(pickup.dt.hour)["is_crawl"].agg(["size", "mean"])
by_weekday = train.groupby(pickup.dt.dayofweek)["is_crawl"].agg(["size", "mean"])

print(by_hour)
print(by_weekday)

                       size      mean
tpep_pickup_datetime                 
0                      6011  0.056729
1                      4033  0.044384
2                      2709  0.026947
3                      1925  0.018182
4                      1647  0.013358
5                      1691  0.010645
6                      3135  0.020096
7                      6044  0.066016
8                      7437  0.139976
9                      8083  0.198936
10                     8497  0.250441
11                     9102  0.287849
12                     9760  0.309119
13                    10126  0.295675
14                    11289  0.300381
15                    11673  0.316542
16                    10952  0.297936
17                    11999  0.285524
18                    12172  0.243756
19                    11120  0.199730
20                    10461  0.126757
21                    11372  0.102708
22                    10828  0.091522
23                     8812  0.070359
            

C:\Users\Diego\AppData\Local\Temp\ipykernel_27260\851161104.py:4: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  by_weekday = train.groupby(pickup.dt.dayofweek)["is_crawl"].agg(["size", "mean"])


In [5]:
by_zone = train.groupby("PULocationID")["is_crawl"].agg(["size", "mean"])
by_zone = by_zone[by_zone["size"] >= 500].sort_values("mean", ascending=False)

by_zone.head(15)

,size,mean
PULocationID,,
161,7746,0.382907
186,5410,0.352126
163,4627,0.331100
162,5693,0.327771
229,3079,0.321858
100,2380,0.321429
230,5246,0.310332
170,4878,0.305658
164,3501,0.294487


In [6]:
RULE_HOURS = range(10, 19)
RULE_ZONES = by_zone.head(10).index.tolist()
print(RULE_ZONES)


def rule_predict(df):
    hour = df["tpep_pickup_datetime"].dt.hour
    return (hour.isin(RULE_HOURS) & df["PULocationID"].isin(RULE_ZONES)).astype(int)


pred_rule = rule_predict(valid)
evaluate("2. rule: hours 10-18 + top zones", y_valid, pred_rule, pred_rule)

[161, 186, 163, 162, 229, 100, 230, 170, 164, 237]


,model,precision,recall,pr_auc,lift
0,1. always no-crawl,0.000000,0.000000,0.190360,1.00000
1,2. rule: hours 10-18 + top zones,0.445129,0.349332,0.279359,1.46753


In [7]:
from sklearn.tree import DecisionTreeClassifier, export_text


def make_X(df):
    return pd.DataFrame({
        "hour": df["tpep_pickup_datetime"].dt.hour,
        "day_of_week": df["tpep_pickup_datetime"].dt.day_of_week,
        "trip_distance": df["trip_distance"],
    })


X_train, y_train = make_X(train), train["is_crawl"]
X_valid = make_X(valid)

tree = DecisionTreeClassifier(max_depth=2, random_state=42)
tree.fit(X_train, y_train)
print(export_text(tree, feature_names=list(X_train.columns)))

score_tree = tree.predict_proba(X_valid)[:, 1]
threshold = y_train.mean()
pred_tree = (score_tree >= threshold).astype(int)

evaluate("3. tree depth 2", y_valid, pred_tree, score_tree)

|--- trip_distance <= 1.92
|   |--- trip_distance <= 0.33
|   |   |--- class: 1
|   |--- trip_distance >  0.33
|   |   |--- class: 0
|--- trip_distance >  1.92
|   |--- trip_distance <= 3.41
|   |   |--- class: 0
|   |--- trip_distance >  3.41
|   |   |--- class: 0



,model,precision,recall,pr_auc,lift
0,1. always no-crawl,0.000000,0.000000,0.190360,1.000000
1,2. rule: hours 10-18 + top zones,0.445129,0.349332,0.279359,1.467530
2,3. tree depth 2,0.314751,0.833421,0.338955,1.780598
